# 1.0 — Setup & Imports

Mount Google Drive and import all required standard-library, third-party, and framework packages.

## 1.1 — Drive Mount

In [1]:
import os
import sys

if 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        IN_COLAB = True
    except Exception as e:
        print(f"Colab drive note: {e}")
        IN_COLAB = False
else:
    IN_COLAB = False


## 1.2 — Library Imports

Groups: standard library → numerical/data → visualisation → deep learning.

In [2]:
# Standard library
import os
import gc
import glob
from pathlib import Path

# Numerical & data
import numpy as np
import pandas as pd
import seaborn as sns

# Visualisation
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Deep learning
import tensorflow as tf
from tensorflow.keras import layers

# Metrics & Splitting
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, f1_score, log_loss
)

# ── Excel export ──────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

sns.set_style("whitegrid")
print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


# 2.0 — Configuration

All tunable constants and filesystem paths live here. Edit this section before running
the notebook on a new dataset or experiment.

## 2.1 — Constants & Paths

In [3]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

import glob

IN_KAGGLE = Path('/kaggle').exists()
IN_COLAB  = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT_DIR  = Path('/kaggle/working/Classification')
    MODEL_DIR = Path('/kaggle/input')
elif IN_COLAB:
    ROOT_DIR  = Path('/content/drive/My Drive/Classification')
    MODEL_DIR = ROOT_DIR / 'ensemble' / 'models' / 'ensembles'
else:
    ROOT_DIR  = Path('.')
    MODEL_DIR = Path('ensemble/models/ensembles')

_data_matches = (
    glob.glob('/kaggle/input/**/k001.csv', recursive=True)
    or glob.glob('/kaggle/input/**/data.csv', recursive=True)
    or glob.glob('Classification/data/data.csv')
    or glob.glob('../data/data.csv')
    or glob.glob('../../Classification/data/data.csv')
    or glob.glob('/content/drive/My Drive/Classification/data/data.csv')
    or glob.glob('data.csv')
)
_ref_matches = (
    glob.glob('/kaggle/input/**/k002.csv', recursive=True)
    or glob.glob('/kaggle/input/**/ref.csv', recursive=True)
    or glob.glob('Classification/data/ref.csv')
    or glob.glob('../data/ref.csv')
    or glob.glob('../../Classification/data/ref.csv')
    or glob.glob('/content/drive/My Drive/Classification/data/ref.csv')
    or glob.glob('ref.csv')
)

DATA_FILE  = Path(_data_matches[0])  if _data_matches else Path('data.csv')
LABEL_FILE = Path(_ref_matches[0])   if _ref_matches  else Path('ref.csv')

# Scene geometry
H, W, B = 330, 307, 6   # height, width, spectral bands
PATCH_SIZE = 9

# Uncertainty thresholds used for masking
AU_THRESH = 0.5
EU_THRESH = 0.2
TU_THRESH = 0.7

print('Data file  :', DATA_FILE)
print('Label file :', LABEL_FILE)


Data file  : /kaggle/input/datasets/daneshselwal/dias-6-bands-data/k001.csv
Label file : /kaggle/input/datasets/daneshselwal/dias-6-bands-data/k002.csv


## 2.2 — Output Directory Setup

Creates a dedicated subfolder inside `MODEL_DIR` to hold all CreDE outputs (PNGs and CSV).

In [4]:
CREDE_OUT_DIR = ROOT_DIR / "ensemble" / "results"
CREDE_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"CreDE output directory: {CREDE_OUT_DIR}")


CreDE output directory: /kaggle/working/Classification/ensemble/results


# 3.0 — Data Loading & Preprocessing

Reads the multispectral image and label rasters from CSV, normalises each band
independently to [0, 1], then extracts every spatial patch needed for full-scene inference.

## 3.1 — Multispectral Data Loader

In [5]:
def load_multispectral_6band(data_path, label_path, height, width, bands):
    """
    Load a flat-CSV multispectral image and its reference label raster.

    Each band is min-max normalised independently to [0, 1].

    Parameters
    ----------
    data_path  : path-like  Path to the pixel-value CSV.
    label_path : path-like  Path to the integer label CSV.
    height, width, bands : int  Scene dimensions.

    Returns
    -------
    x_norm : np.ndarray  Shape (H, W, B), dtype float32, values in [0, 1].
    y      : np.ndarray  Shape (H, W),    dtype int32.
    """
    x = pd.read_csv(data_path).to_numpy(dtype=np.float32).reshape(height, width, bands)
    y = pd.read_csv(label_path).to_numpy(dtype=np.int32).reshape(height, width)

    x_norm = np.empty_like(x, dtype=np.float32)
    for b in range(bands):
        band = x[:, :, b]
        band_min, band_max = np.min(band), np.max(band)
        x_norm[:, :, b] = (band - band_min) / max(band_max - band_min, 1e-8)

    return x_norm, y


print("Loading multispectral data...")
x_img, y_img = load_multispectral_6band(DATA_FILE, LABEL_FILE, H, W, B)
num_classes = int(np.unique(y_img[y_img > 0]).size)
print(f"Scene shape: {x_img.shape}  |  Classes: {num_classes}")

Loading multispectral data...
Scene shape: (330, 307, 6)  |  Classes: 7


## 3.2 — Spatial Patch Extraction

Pads the scene and slides a `PATCH_SIZE × PATCH_SIZE` window over every pixel,
producing a flat array of patches ready for batch inference.

In [6]:
print("Extracting all spatial patches for full-scene inference...")

pad   = PATCH_SIZE // 2
x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode="edge")

scene_pixels_scaled = np.empty((H * W, PATCH_SIZE, PATCH_SIZE, B), dtype=np.float32)

idx = 0
for r in range(H):
    for c in range(W):
        scene_pixels_scaled[idx] = x_pad[r:r + PATCH_SIZE, c:c + PATCH_SIZE, :]
        idx += 1

print(f"scene_pixels_scaled shape: {scene_pixels_scaled.shape}")

Extracting all spatial patches for full-scene inference...
scene_pixels_scaled shape: (101310, 9, 9, 6)


In [7]:
# -----------------------------
# 3.3 — Labeled Patch Extraction & Splitting
# -----------------------------
print("Extracting labeled patches for metric evaluation...")

def extract_labeled_patches(x, y, patch_size=9):
    pad = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")
    coords = np.argwhere(y > 0)
    patches = np.empty((coords.shape[0], patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels = np.empty((coords.shape[0],), dtype=np.int32)
    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i] = int(y[r, c]) - 1
    return patches, labels, coords

X, y_labels, coords = extract_labeled_patches(x_img, y_img, PATCH_SIZE)

# Standard Split
TRAIN_PERCENT = 0.75
x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y_labels, train_size=TRAIN_PERCENT, random_state=SEED, stratify=y_labels
)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

print(f"Standard Test Set Extracted: {x_test.shape}")

# AlexNet legacy split — the AlexNet ensemble members were trained with this split
# (seed 10), so their held-out test pixels are different from the standard split.
ALEXNET_LEGACY_SPLIT_SEED    = 10
ALEXNET_LEGACY_TRAIN_PERCENT = 0.75
_, x_test_alex, _, y_test_alex = train_test_split(
    X, y_labels,
    train_size=ALEXNET_LEGACY_TRAIN_PERCENT,
    random_state=ALEXNET_LEGACY_SPLIT_SEED,
    stratify=y_labels
)
print(f"AlexNet Legacy Test Set Extracted: {x_test_alex.shape}")

Extracting labeled patches for metric evaluation...
Standard Test Set Extracted: (4310, 9, 9, 6)
AlexNet Legacy Test Set Extracted: (4310, 9, 9, 6)


# 4.0 — Custom Keras Layers

Registers the four custom `tf.keras` layers required to deserialise the saved `.keras` models.
All four are decorated with `@register_keras_serializable()` so they survive `load_model`.
**Do not alter any layer internals** — these must match the definitions used during training.

## 4.1 — PatchExtractor

In [8]:
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    """Extracts non-overlapping image patches and flattens them into a sequence."""

    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )
        batch, patch_dim = tf.shape(images)[0], tf.shape(patches)[-1]
        num_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        return tf.reshape(patches, [batch, num_patches, patch_dim])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size})
        return cfg

## 4.2 — PatchPositionEncoder

In [9]:
@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    """Projects patches to `projection_dim` and adds learned positional embeddings."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg

## 4.3 — GlobalFilterLayer

In [10]:
@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    """
    Applies a learnable complex-valued filter in the 2-D frequency domain.

    The token sequence is reshaped to a square grid, transformed with FFT2D,
    element-wise multiplied by trainable complex weights, then inverse-transformed
    back to the spatial sequence.
    """

    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(
            name="w_real", shape=(self.token_side, self.token_side, channels),
            initializer="glorot_uniform", trainable=True
        )
        self.w_imag = self.add_weight(
            name="w_imag", shape=(self.token_side, self.token_side, channels),
            initializer="zeros", trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch    = tf.shape(x)[0]
        channels = tf.shape(x)[-1]

        x_2d       = tf.reshape(x, [batch, self.token_side, self.token_side, channels])
        x_fft      = tf.signal.fft2d(tf.cast(x_2d, tf.complex64))
        w_complex  = tf.complex(self.w_real, self.w_imag)
        x_filtered = x_fft * w_complex
        x_spatial  = tf.math.real(tf.signal.ifft2d(x_filtered))

        return tf.reshape(x_spatial, [batch, self.token_side * self.token_side, channels])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"token_side": self.token_side})
        return cfg

## 4.4 — PatchEncoderWithCLS

In [11]:
@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    """
    Projects patches, prepends a learnable [CLS] token, and adds positional embeddings.

    Used by the ViT-UNet architecture; the CLS token aggregates global context
    for the final classification head.
    """

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name="cls_token", shape=(1, 1, self.projection_dim),
            initializer="zeros", trainable=True
        )
        super().build(input_shape)

    def call(self, patches):
        batch      = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)  # broadcast CLS to batch
        x          = tf.concat([cls_tokens, patch_proj], axis=1)
        positions  = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return cfg

## 4.5 — Custom Objects Registry

Collects all custom layer classes into a single dict passed to `tf.keras.models.load_model`.

In [12]:
CUSTOM_OBJECTS = {
    "PatchExtractor":      PatchExtractor,
    "PatchPositionEncoder": PatchPositionEncoder,
    "GlobalFilterLayer":   GlobalFilterLayer,
    "PatchEncoderWithCLS": PatchEncoderWithCLS,
}

# 5.0 — Core Ensemble Functions

Functions for discovering saved ensemble checkpoints and running the CreDE
credal-bound computation across all ensemble members.

## 5.1 — Ensemble Path Discovery

In [13]:
def _dedup_by_filename(paths):
    """Keep one path per unique filename (shallowest path first), sorted by filename."""
    seen = set()
    unique = []
    for p in sorted(paths, key=lambda x: (len(Path(x).parts), str(x))):
        name = Path(p).name
        if name not in seen:
            seen.add(name)
            unique.append(str(p))
    return sorted(unique, key=lambda x: Path(x).name)


def get_ensemble_paths(model_name):
    """
    Locate all 5 saved ensemble checkpoint files for a given architecture.
    Searches /kaggle/input recursively (including cuq6-ensemble-k04) and deduplicates by filename.
    """
    candidates = [
        f"/kaggle/input/cuq6-ensemble-k04/**/{model_name}_ens_*_final.keras",
        f"/kaggle/input/notebooks/daneshselwal/cuq6-ensemble-k04/**/{model_name}_ens_*_final.keras",
        f"/kaggle/input/**/{model_name}_ens_*_final.keras",
        str(MODEL_DIR / f"{model_name}_ens_*_final.keras"),
        str(MODEL_DIR / "ensembles" / f"{model_name}_ens_*_final.keras"),
        str(MODEL_DIR / f"ensembles_{model_name}" / f"{model_name}_ens_*_final.keras"),
        str(ROOT_DIR / "ensemble" / "models" / f"{model_name}_ens_*_final.keras"),
        str(ROOT_DIR / "ensemble" / "models" / "ensembles" / f"{model_name}_ens_*_final.keras"),
        f"/kaggle/working/**/{model_name}_ens_*_final.keras",
        f"**/{model_name}_ens_*_final.keras",
    ]
    for pattern in candidates:
        paths = glob.glob(pattern, recursive=True)
        if paths:
            return _dedup_by_filename(paths)
    return []


## 5.2 — Homogeneous Ensemble Evaluation (CreDE)

Loads each ensemble member sequentially, predicts on the full scene,
then computes credal bounds (p_min / p_max) to derive aleatoric uncertainty (AU),
epistemic uncertainty (EU), and total uncertainty (TU).
Each model is deleted immediately after prediction to minimise peak RAM usage.

In [14]:
def evaluate_homogeneous_ensemble(model_paths, input_data, batch_size=2048):
    """
    Run CreDE inference over a list of ensemble checkpoints.

    For each model: predict → stack → compute credal bounds → derive uncertainty measures.

    Parameters
    ----------
    model_paths : list[str]   Paths to `.keras` checkpoint files.
    input_data  : np.ndarray  Full-scene patch array, shape (N, P, P, B).
    batch_size  : int         Inference batch size (default 2048).

    Returns
    -------
    pred_class : np.ndarray  Shape (N,)   — argmax of credal mean probabilities.
    p_star     : np.ndarray  Shape (N, C) — normalised lower credal probabilities.
    au         : np.ndarray  Shape (N,)   — aleatoric uncertainty (entropy of p_star).
    eu         : np.ndarray  Shape (N,)   — epistemic uncertainty (mean credal spread).
    tu         : np.ndarray  Shape (N,)   — total uncertainty (au + eu).
    """
    all_preds = []

    for path in model_paths:
        print(f"  -> Loading & predicting: {Path(path).name}")
        model = tf.keras.models.load_model(
            path, compile=False, custom_objects=CUSTOM_OBJECTS, safe_mode=False
        )
        preds = model.predict(input_data, batch_size=batch_size, verbose=0)
        all_preds.append(preds)

        # Delete model immediately to free RAM before loading the next member
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    print("  -> Computing credal bounds...")
    stacked_preds = tf.stack(all_preds, axis=0)  # (M, N, C)
    p_min = tf.reduce_min(stacked_preds, axis=0)
    p_max = tf.reduce_max(stacked_preds, axis=0)

    delta_p = p_max - p_min

    # Normalise lower credal probabilities to form p_star
    p_star = p_min / (tf.reduce_sum(p_min, axis=-1, keepdims=True) + 1e-12)
    p_star = np.clip(p_star, 1e-12, 1.0)

    # Uncertainty decomposition
    au = -np.sum(p_star * np.log(p_star), axis=-1)   # aleatoric: entropy
    eu = np.mean(delta_p, axis=-1)                    # epistemic: mean credal spread
    tu = au + eu                                      # total uncertainty

    pred_class = np.argmax(p_star, axis=-1)

    # Coerce any TF tensors to numpy
    to_np = lambda t: t.numpy() if hasattr(t, 'numpy') else t
    return to_np(pred_class), to_np(p_star), to_np(au), to_np(eu), to_np(tu)

# 6.0 — Visualisation Engine

Generates and saves a standardised 6-panel figure for each architecture:
base prediction map, binary certain/uncertain mask, pixel-count bar chart,
and three grey-overlay maps (AU, EU, TU).

## 6.1 — 6-Panel Spatial Mapping Function

In [15]:
# Shared colour palette (up to 10 classes + grey for uncertain pixels)
CLASS_COLORS = [
    '#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF',
    '#FFFF00', '#A52A2A', '#FFA500', '#7FFF00', '#8A2BE2'
]

def generate_spatial_crede_maps(
    model_name, pred_class_scene, p_star_scene, au_scene, eu_scene, tu_scene,
    H=330, W=307, au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7
):
    """Produce and save a 3x4 spatial uncertainty figure for CreDE."""
    print(f'  -> Generating 3x4 spatial maps for {model_name}...')
    n_cls = p_star_scene.shape[-1]

    au_map         = au_scene.reshape((H, W))
    eu_map         = eu_scene.reshape((H, W))
    tu_map         = tu_scene.reshape((H, W))
    pred_map       = pred_class_scene.reshape((H, W))

    au_mask = (au_map > au_thresh).astype(int)
    eu_mask = (eu_map > eu_thresh).astype(int)
    tu_mask = (tu_map > tu_thresh).astype(int)

    combined_au = np.where(au_mask == 1, n_cls, pred_map)
    combined_eu = np.where(eu_mask == 1, n_cls, pred_map)
    combined_tu = np.where(tu_mask == 1, n_cls, pred_map)

    cmap_base   = ListedColormap(CLASS_COLORS[:n_cls])
    cmap_unc    = ListedColormap(CLASS_COLORS[:n_cls] + ['#808080'])
    cmap_binary = ListedColormap(['#FFFF00', '#001F3F'])

    bar_lbls = [f'Class {i}' for i in range(n_cls)] + ['Uncertain']
    bar_cols = CLASS_COLORS[:n_cls] + ['#808080']

    fig, axes = plt.subplots(3, 4, figsize=(38, 26))
    fig.suptitle(f'{model_name} — CreDE Uncertainty Maps (Absolute Thresholds)',
                 fontsize=24, fontweight='bold', y=0.99)

    # ── Row 0: Base prediction + 3 binary maps
    axes[0, 0].imshow(pred_map, cmap=cmap_base, vmin=0, vmax=n_cls - 1)
    axes[0, 0].set_title('Base Prediction Map', fontsize=15)
    axes[0, 0].axis('off')

    binary_specs = [
        (axes[0, 1], au_mask, f'Aleatoric (AU > {au_thresh})'),
        (axes[0, 2], eu_mask, f'Epistemic (EU > {eu_thresh})'),
        (axes[0, 3], tu_mask, f'Total      (TU > {tu_thresh})'),
    ]
    for ax, mask, label in binary_specs:
        ax.imshow(mask, cmap=cmap_binary, vmin=0, vmax=1)
        ax.set_title(f'Certain vs Uncertain\n{label}', fontsize=15, pad=10)
        ax.axis('off')
        ax.legend(
            handles=[Patch(facecolor='#FFFF00', label='Certain'),
                     Patch(facecolor='#001F3F', label='Uncertain')],
            loc='upper left', bbox_to_anchor=(0.0, -0.02), borderaxespad=0,
            fontsize=11, framealpha=0.9, ncol=2
        )

    # ── Row 1: Grey overlay maps (AU, EU, TU) + blank
    overlay_specs = [
        (axes[1, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[1, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[1, 2], combined_tu, f'Total      (TU > {tu_thresh})'),
    ]
    for ax, combined, label in overlay_specs:
        ax.imshow(combined, cmap=cmap_unc, vmin=0, vmax=n_cls)
        ax.set_title(f'Grey Overlay — {label}', fontsize=15, pad=10)
        ax.axis('off')

    axes[1, 3].axis('off')

    # ── Row 2: Bar charts (AU, EU, TU) + blank
    bar_specs = [
        (axes[2, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[2, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[2, 2], combined_tu, f'Total      (TU > {tu_thresh})'),
    ]
    for ax, combined, label in bar_specs:
        uniq, cnt = np.unique(combined, return_counts=True)
        c_dict   = {int(k): int(v) for k, v in zip(uniq, cnt)}
        bar_vals = [c_dict.get(i, 0) for i in range(n_cls + 1)]
        ax.bar(bar_lbls, bar_vals, color=bar_cols, edgecolor='black')
        ax.set_title(f'Pixel Counts — {label}', fontsize=15, pad=10)
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.set_ylabel('Pixel Count', fontsize=12)
        max_val = max(bar_vals, default=1)
        for i, v in enumerate(bar_vals):
            ax.text(i, v + max_val * 0.01,
                    f'{v:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
        ax.set_ylim(0, max_val * 1.12)

    axes[2, 3].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_path = CREDE_OUT_DIR / f'{model_name}_CreDE_spatial_maps.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  -> Saved: {save_path}')
    return str(save_path)

# 7.0 — Main Execution

Iterates over each target architecture, runs CreDE inference, generates spatial maps,
and accumulates per-architecture metrics into a master summary CSV.
RAM is wiped aggressively after every architecture to prevent OOM errors.

In [16]:
# ── Excel Export Helpers ──────────────────────────────────────────────────────
_HDR_FILL   = PatternFill('solid', start_color='1F4E79')
_ALT_FILL   = PatternFill('solid', start_color='D6E4F0')
_HDR_FONT   = Font(name='Arial', bold=True, color='FFFFFF', size=11)
_BODY_FONT  = Font(name='Arial', size=10)
_TITLE_FONT = Font(name='Arial', bold=True, size=13)
_CENTER     = Alignment(horizontal='center', vertical='center', wrap_text=True)
_LEFT       = Alignment(horizontal='left',   vertical='center')
_THIN_SIDE  = Side(style='thin', color='AAAAAA')
_THIN_BORDER= Border(left=_THIN_SIDE, right=_THIN_SIDE,
                     top=_THIN_SIDE,  bottom=_THIN_SIDE)

def _style_header_row(ws, row, col_start, col_end):
    for c in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill   = _HDR_FILL
        cell.font   = _HDR_FONT
        cell.alignment = _CENTER
        cell.border = _THIN_BORDER

def _style_data_rows(ws, row_start, row_end, col_start, col_end):
    for r in range(row_start, row_end + 1):
        fill = _ALT_FILL if r % 2 == 0 else PatternFill()
        for c in range(col_start, col_end + 1):
            cell = ws.cell(row=r, column=c)
            cell.font      = _BODY_FONT
            cell.fill      = fill
            cell.alignment = _LEFT
            cell.border    = _THIN_BORDER

def _write_df_to_sheet(ws, df, start_row=1, start_col=1, title=None):
    r = start_row
    if title:
        cell = ws.cell(row=r, column=start_col, value=title)
        cell.font      = _TITLE_FONT
        cell.alignment = _LEFT
        _end_col = start_col + len(df.columns) - 1
        if _end_col >= start_col:
            ws.merge_cells(start_row=r, start_column=start_col,
                           end_row=r, end_column=_end_col)
        r += 1

    for j, col_name in enumerate(df.columns, start=start_col):
        ws.cell(row=r, column=j, value=col_name)
    _style_header_row(ws, r, start_col, start_col + len(df.columns) - 1)
    r += 1

    data_start = r
    for _, row_data in df.iterrows():
        for j, val in enumerate(row_data, start=start_col):
            ws.cell(row=r, column=j, value=round(float(val), 6)
                    if isinstance(val, (float, np.floating)) else val)
        r += 1
    _style_data_rows(ws, data_start, r - 1, start_col, start_col + len(df.columns) - 1)

    for j, col_name in enumerate(df.columns, start=start_col):
        max_len = max(len(str(col_name)),
                      max((len(str(ws.cell(row=row, column=j).value or ''))
                           for row in range(data_start, r)), default=0))
        ws.column_dimensions[get_column_letter(j)].width = min(max_len + 4, 40)
    return r

def create_crede_excel_report(out_dir, summary_df, plot_paths,
                              credal_overall_df=None, credal_classwise_df=None,
                              pixel_uncertainty=None):
    """Auto-creates CreDE_Results.xlsx with Summary, Credal Coverage, Plots, Uncertain Pixels, and Threshold Analysis."""
    xlsx_path = Path(out_dir) / 'CreDE_Results.xlsx'
    wb = Workbook()

    # ── Sheet 1: Summary ──────────────────────────────────────────────────────
    ws1 = wb.active
    ws1.title = 'CreDE Summary'
    ws1.row_dimensions[1].height = 30
    _write_df_to_sheet(ws1, summary_df, start_row=1, title='CreDE — Inference Summary')

    # ── Credal coverage sheet (overall coverage, avg set size, class-wise coverage) ──
    if credal_overall_df is not None and credal_classwise_df is not None:
        add_credal_coverage_sheet(
            wb, credal_overall_df, credal_classwise_df,
            bounds_note='Credal bounds per pixel: lower = min over the 5 ensemble members, upper = max over the 5 members (CreDE).',
            index=1)

    # ── Sheet 2: Plots ────────────────────────────────────────────────────────
    if plot_paths:
        ws2 = wb.create_sheet('Plots')
        title_cell = ws2.cell(row=1, column=1, value='CreDE — Spatial Uncertainty Maps')
        title_cell.font = Font(name='Arial', bold=True, size=14)

        img_row = 3
        for label, img_path in plot_paths:
            if not Path(img_path).exists():
                continue
            ws2.cell(row=img_row, column=1, value=label).font = Font(name='Arial', bold=True, size=11)
            img_row += 1

            xl_img = XLImage(img_path)
            orig_w, orig_h = xl_img.width, xl_img.height
            target_w = 900
            scale    = target_w / orig_w if orig_w > 0 else 1
            xl_img.width  = int(orig_w * scale)
            xl_img.height = int(orig_h * scale)

            ws2.add_image(xl_img, f'A{img_row}')
            ws2.row_dimensions[img_row].height = xl_img.height * 0.75
            img_row += int(xl_img.height / 15) + 3

    # ── Extra sheets: per-pixel AU / EU / TU + threshold analysis ──────────────
    if pixel_uncertainty:
        add_pixel_uncertainty_sheets(wb, pixel_uncertainty)

    wb.save(xlsx_path)
    print(f'\n✅ Excel report saved → {xlsx_path}')
    return str(xlsx_path)

save_crede_excel_report = create_crede_excel_report


In [17]:
# -----------------------------
# Metric Helpers
# -----------------------------
def multiclass_brier_score(y_onehot, y_prob):
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, y_prob, n_bins=15):
    confidences, predictions = np.max(y_prob, axis=1), np.argmax(y_prob, axis=1)
    correct = (predictions == y_true).astype(np.float32)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (confidences >= bin_edges[i]) & (confidences <= bin_edges[i+1] if i == n_bins - 1 else confidences < bin_edges[i+1])
        prop = np.mean(in_bin)
        if prop > 0:
            ece += np.abs(np.mean(correct[in_bin]) - np.mean(confidences[in_bin])) * prop
    return float(ece)

## 7.0a — Credal Prediction-Set Metrics

Coverage-style metrics for CreDE, defined exactly like the conformal-prediction ones: overall empirical coverage, average set size and class-wise empirical coverage. Bounds: `lower = min over ensemble members`, `upper = max over ensemble members`. Evaluated on the labelled held-out test pixels.

In [18]:
# ── Credal prediction-set metrics (interval dominance) ─────────────────────────
def credal_set_metrics(lower, upper, y_true, num_classes):
    """
    Turn credal probability bounds into prediction SETS and score them like
    conformal prediction does (overall coverage, average set size, class-wise coverage).

    Set rule (interval dominance): class k stays in C(x) unless another class is
    certainly better, i.e.  k in C(x)  <=>  upper_k(x) >= max_j lower_j(x).
    The argmax-of-lower class is always kept, so no set is ever empty.

    lower, upper : (N, C) credal bounds     y_true : (N,) integer labels 0..C-1
    """
    lower  = np.asarray(lower,  dtype=np.float64)
    upper  = np.maximum(np.asarray(upper, dtype=np.float64), lower)
    y_true = np.asarray(y_true).astype(int)
    n      = len(y_true)

    in_set  = upper >= lower.max(axis=1, keepdims=True)      # (N, C) boolean sets
    covered = in_set[np.arange(n), y_true]                   # true label inside set?

    classwise = np.array([
        covered[y_true == k].mean() if np.any(y_true == k) else np.nan
        for k in range(num_classes)
    ])
    return {
        'n':         int(n),
        'coverage':  float(covered.mean()),                  # overall empirical coverage
        'avg_size':  float(in_set.sum(axis=1).mean()),       # average set size
        'classwise': classwise,                              # class-wise empirical coverage
    }


def credal_frames(records):
    """records: list of (model_name, metrics_dict) -> (overall_df, classwise_df)."""
    overall = pd.DataFrame([{
        'Model':                      m,
        'N_Test_Pixels':              r['n'],
        'Overall_Empirical_Coverage': r['coverage'],
        'Average_Set_Size':           r['avg_size'],
        'Mean_Class_Coverage':        float(np.nanmean(r['classwise'])),
    } for m, r in records])
    classwise = pd.DataFrame([{
        'Model': m,
        **{f'Class_{k + 1}_Coverage': float(v) for k, v in enumerate(r['classwise'])},
    } for m, r in records])
    return overall, classwise


def add_credal_coverage_sheet(wb, overall_df, classwise_df, bounds_note, index=1):
    """Add a 'Credal Coverage' sheet (overall + class-wise tables + method notes)."""
    ws = wb.create_sheet('Credal Coverage', index)
    r = _write_df_to_sheet(ws, overall_df, start_row=1,
                           title='Credal prediction sets — overall empirical coverage & average set size') + 1
    r = _write_df_to_sheet(ws, classwise_df, start_row=r,
                           title='Credal prediction sets — class-wise empirical coverage') + 1
    notes = [
        'How the numbers are computed (same definitions as for conformal prediction):',
        bounds_note,
        'Set rule (interval dominance): class k is in the set  iff  upper_k >= max_j lower_j.',
        'Overall coverage = fraction of labelled test pixels whose true class is inside the set.',
        'Average set size = mean number of classes in the set (1 = confident singleton).',
        'Class-wise coverage (class k) = fraction of test pixels of true class k whose set contains k.',
        'Class_k = reference label k (1-based, same as the reference raster; 0 = unlabelled, excluded).',
        'Coverage here is MEASURED on the test set; unlike (split) conformal prediction it is not calibrated to a target 1-alpha.',
    ]
    for line in notes:
        ws.cell(row=r, column=1, value=line).font = Font(name='Arial', size=10, bold=(line == notes[0]))
        r += 1
    return ws


def predict_ensemble_credal_bounds(model_paths, x_data, batch_size=1024):
    """
    Run every model in the ensemble over x_data and return (p_min, p_max).
    Loads models sequentially to stay well within GPU VRAM / host RAM limits.
    """
    all_p = []
    for p in model_paths:
        m = tf.keras.models.load_model(
            p, compile=False, custom_objects=CUSTOM_OBJECTS, safe_mode=False
        )
        preds = m.predict(x_data, batch_size=batch_size, verbose=0)
        all_p.append(preds)
        del m
        tf.keras.backend.clear_session()
        gc.collect()

    stacked = np.stack(all_p, axis=0)          # (M, N, C)
    return stacked.min(axis=0), stacked.max(axis=0)


# ── Pixel-level uncertainty table + threshold analysis (for the Excel report) ──
EXCEL_MAX_DATA_ROWS    = 1_048_575            # Excel limit (1,048,576 rows) minus the header row
SENSITIVITY_THRESHOLDS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]


def build_pixel_uncertainty_df(unc, y_ref):
    """
    One row per pixel that is uncertain under AU, EU or TU (union of the three masks).
    Row / Col / Pixel_ID are 1-based (Pixel_ID = (Row-1)*W + Col, row-major).
    Predicted_Class is 1-based. Reference_Class is the reference label (0 = unlabelled).
    Sorted by TU, highest first.
    """
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    W = unc['W']
    au_t, eu_t, tu_t = unc['thresholds']

    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t
    idx = np.flatnonzero(f_au | f_eu | f_tu)

    r0, c0 = np.divmod(idx, W)
    tags = np.char.add(
        np.char.add(np.where(f_au[idx], 'AU ', ''), np.where(f_eu[idx], 'EU ', '')),
        np.where(f_tu[idx], 'TU', '')
    )
    under = pd.Series(tags).str.strip().str.replace(' ', '+', regex=False)

    df = pd.DataFrame({
        'Pixel_ID':        idx + 1,
        'Row':             r0 + 1,
        'Col':             c0 + 1,
        'Predicted_Class': unc['pred'][idx] + 1,
        'Reference_Class': np.asarray(y_ref).reshape(-1)[idx],
        'AU':              au[idx].astype(np.float64).round(6),
        'EU':              eu[idx].astype(np.float64).round(6),
        'TU':              tu[idx].astype(np.float64).round(6),
        f'AU_Uncertain (>{au_t})': f_au[idx].astype(int),
        f'EU_Uncertain (>{eu_t})': f_eu[idx].astype(int),
        f'TU_Uncertain (>{tu_t})': f_tu[idx].astype(int),
        'Uncertain_Under': under.values,
    })
    return df.sort_values('TU', ascending=False, kind='mergesort').reset_index(drop=True)


def build_threshold_analysis(unc):
    """Summary tables: is AU+EU=TU, and how much do the thresholds matter?"""
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    n = au.size
    au_t, eu_t, tu_t = unc['thresholds']
    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t

    counts = pd.DataFrame({
        'Criterion':        ['AU', 'EU', 'TU'],
        'Threshold':        [au_t, eu_t, tu_t],
        'Uncertain_Pixels': [int(f_au.sum()), int(f_eu.sum()), int(f_tu.sum())],
    })
    counts['Percent_of_Scene'] = (100.0 * counts['Uncertain_Pixels'] / n).round(3)

    regions = [
        ('AU only', 1, 0, 0), ('EU only', 0, 1, 0), ('TU only', 0, 0, 1),
        ('AU & EU only', 1, 1, 0), ('AU & TU only', 1, 0, 1), ('EU & TU only', 0, 1, 1),
        ('AU & EU & TU', 1, 1, 1), ('None (certain under all three)', 0, 0, 0),
    ]
    overlap = pd.DataFrame([
        {'Region (mutually exclusive)': name,
         'Pixels': int(((f_au == bool(a)) & (f_eu == bool(e)) & (f_tu == bool(t))).sum())}
        for name, a, e, t in regions
    ])

    au_vs_tu = pd.DataFrame({
        'Comparison': ['AU-uncertain but NOT TU-uncertain', 'TU-uncertain but NOT AU-uncertain'],
        'Pixels':     [int((f_au & ~f_tu).sum()), int((f_tu & ~f_au).sum())],
    })

    diff = np.abs(au.astype(np.float64) + eu.astype(np.float64) - tu.astype(np.float64))
    identity = pd.DataFrame({
        'Metric': ['max |AU + EU - TU|', 'mean |AU + EU - TU|', 'min (TU - AU)  (= min EU, must be >= 0)',
                   'max possible AU = ln(#classes)', 'max observed AU', 'max observed EU', 'max observed TU'],
        'Value':  [float(diff.max()), float(diff.mean()), float((tu - au).min()),
                   float(np.log(unc['n_classes'])), float(au.max()), float(eu.max()), float(tu.max())],
    })

    sens = pd.DataFrame({'Threshold': SENSITIVITY_THRESHOLDS})
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (pixels)'] = [int((arr > t).sum()) for t in SENSITIVITY_THRESHOLDS]
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (%)'] = [round(100.0 * float((arr > t).mean()), 3) for t in SENSITIVITY_THRESHOLDS]

    return {'counts': counts, 'overlap': overlap, 'au_vs_tu': au_vs_tu,
            'identity': identity, 'sensitivity': sens, 'n_scene_pixels': int(n)}


def prepare_pixel_uncertainty(model_name, unc, y_ref, results_dir):
    """Build the uncertain-pixel table (+ full CSV copy) and threshold analysis for one model."""
    df_full  = build_pixel_uncertainty_df(unc, y_ref)
    csv_path = Path(results_dir) / f'{model_name}_uncertain_pixels.csv'
    df_full.to_csv(csv_path, index=False)

    truncated = len(df_full) > EXCEL_MAX_DATA_ROWS
    df_xl     = df_full.iloc[:EXCEL_MAX_DATA_ROWS] if truncated else df_full
    print(f'  [{model_name}] uncertain pixels (AU|EU|TU): {len(df_full):,}  → full table: {csv_path}')
    if truncated:
        print(f'  ⚠️ [{model_name}] exceeds Excel row limit; sheet holds the top {len(df_xl):,} by TU. '
              f'Use the CSV for the rest.')
    return {
        'df': df_xl, 'n_uncertain_total': len(df_full), 'truncated': truncated,
        'csv_path': str(csv_path), 'analysis': build_threshold_analysis(unc),
        'thresholds': unc['thresholds'],
    }


def add_pixel_uncertainty_sheets(wb, pixel_uncertainty):
    """Add one 'Uncertain Pixels - <model>' sheet per model + one 'Threshold Analysis' sheet."""
    if not pixel_uncertainty:
        return
    for mname, res in pixel_uncertainty.items():
        ws = wb.create_sheet(f'Uncertain Pixels - {mname}'[:31])
        df = res['df']
        ws.append(list(df.columns))
        _style_header_row(ws, 1, 1, len(df.columns))
        ws.row_dimensions[1].height = 32
        for row in df.itertuples(index=False, name=None):
            ws.append(row)
        for j, col in enumerate(df.columns, start=1):
            ws.column_dimensions[get_column_letter(j)].width = max(12, min(len(str(col)) + 4, 26))
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = f'A1:{get_column_letter(len(df.columns))}{max(len(df) + 1, 2)}'

    wa = wb.create_sheet('Threshold Analysis')
    wa.cell(row=1, column=1, value='CreDE — Uncertain-pixel analysis (AU / EU / TU)').font = Font(
        name='Arial', bold=True, size=14)

    first = next(iter(pixel_uncertainty.values()))
    au_t, eu_t, tu_t = first['thresholds']
    notes = [
        f'Thresholds used: AU > {au_t}, EU > {eu_t}, TU > {tu_t}.',
        'TU is computed as TU = AU + EU in this pipeline, so the identity holds for every pixel (see "Identity check").',
        'Because EU >= 0, TU >= AU for every pixel. Pixel COUNTS can still differ per criterion because each uses its own threshold.',
        'AU is an entropy in nats (max = ln #classes); EU is a mean credal spread in [0, 1]. Equal thresholds are not directly comparable.',
        'Row / Col / Pixel_ID are 1-based. Reference_Class 0 = unlabelled pixel. Pixel sheets list pixels uncertain under AU, EU or TU (sorted by TU).',
    ]
    r = 3
    for line in notes:
        wa.cell(row=r, column=1, value=line).font = _BODY_FONT
        r += 1
    r += 1

    for mname, res in pixel_uncertainty.items():
        a = res['analysis']
        wa.cell(row=r, column=1, value=f'{mname}').font = Font(name='Arial', bold=True, size=13, color='1F4E79')
        r += 1
        wa.cell(row=r, column=1,
                value=f"Scene pixels: {a['n_scene_pixels']:,}   |   Uncertain under AU, EU or TU: {res['n_uncertain_total']:,}"
                      + ('   |   ⚠️ pixel sheet truncated to Excel row limit (see CSV)' if res['truncated'] else '')
                ).font = _BODY_FONT
        r += 2
        r = _write_df_to_sheet(wa, a['counts'],      start_row=r, title='Pixels flagged at the chosen thresholds') + 1
        r = _write_df_to_sheet(wa, a['overlap'],     start_row=r, title='Overlap between AU, EU and TU masks') + 1
        r = _write_df_to_sheet(wa, a['au_vs_tu'],    start_row=r, title='AU vs TU') + 1
        r = _write_df_to_sheet(wa, a['identity'],    start_row=r, title='Identity check  AU + EU = TU') + 1
        r = _write_df_to_sheet(wa, a['sensitivity'], start_row=r, title='Threshold sensitivity (same threshold t applied to AU, EU and TU)') + 2

    wa.column_dimensions['A'].width = 44
    for col in 'BCDEFGHIJ':
        wa.column_dimensions[col].width = 20


## 7.1 — Master Evaluation Loop & Summary Export

In [19]:
master_results = []
plot_entries = []
credal_records = []   # (model_name, credal_set_metrics dict) for the coverage sheet
scene_uncertainty = {} # per-model uncertainty dicts for pixel-level uncertainty

# Assuming you want to loop over these architectures (edit if you have more)
architectures = ['AlexNet_CNN', 'GFNet', 'ViT_UNet']

for model_name in architectures:
    print(f"\n{'='*60}\n  Evaluating CreDE: {model_name}\n{'='*60}")

    ensemble_paths = get_ensemble_paths(model_name)
    if not ensemble_paths:
        print(f"  -> No ensemble models found for {model_name}. Skipping.")
        continue

    # Step 1: Run homogenous ensemble evaluation
    pred_class, p_star, au, eu, tu = evaluate_homogeneous_ensemble(
        ensemble_paths, scene_pixels_scaled, batch_size=2048
    )

    # Step 1b: Credal prediction-set metrics on the labelled held-out test pixels
    #          (AlexNet members use the legacy seed-10 split, GFNet/ViT the standard split)
    x_te_c, y_te_c = (x_test_alex, y_test_alex) if model_name == 'AlexNet_CNN' else (x_test, y_test)
    p_lo, p_hi = predict_ensemble_credal_bounds(ensemble_paths, x_te_c, batch_size=1024)
    credal_m = credal_set_metrics(p_lo, p_hi, y_te_c, num_classes)
    credal_records.append((f'{model_name}_CreDE', credal_m))
    print(f"  -> Credal coverage: {credal_m['coverage']:.4f} | avg set size: {credal_m['avg_size']:.3f}")
    del p_lo, p_hi

    # Step 1c: Store scene uncertainty for pixel-level export
    scene_uncertainty[model_name] = {
        'au':         au,
        'eu':         eu,
        'tu':         tu,
        'pred':       pred_class.reshape(-1),
        'H':          H,
        'W':          W,
        'n_classes':  num_classes,
        'thresholds': (AU_THRESH, EU_THRESH, TU_THRESH),
    }

    # Step 2: Generate 3x4 spatial maps
    saved_plot_path = generate_spatial_crede_maps(
        model_name, pred_class, p_star, au, eu, tu,
        H=H, W=W, au_thresh=AU_THRESH, eu_thresh=EU_THRESH, tu_thresh=TU_THRESH
    )
    plot_entries.append((f'{model_name} — Spatial Uncertainty Maps', saved_plot_path))

    # Step 3: Accumulate summary metrics
    unique, counts = np.unique(pred_class, return_counts=True)
    pixel_counts   = dict(zip(unique, counts))

    master_results.append({
        "Model":    f"{model_name}_CreDE",
        "Mean_AU":  float(np.mean(au)),
        "Mean_EU":  float(np.mean(eu)),
        "Mean_TU":  float(np.mean(tu)),
        "Credal_Coverage":     credal_m["coverage"],
        "Credal_Avg_Set_Size": credal_m["avg_size"],
        **{f"Class_{int(k)}_Pixels": int(v) for k, v in pixel_counts.items()}
    })

    # Step 4: Aggressively free RAM before next architecture
    del pred_class, p_star, au, eu, tu
    tf.keras.backend.clear_session()
    gc.collect()

# --- Export master summary & Excel Report ---
if master_results:
    df_summary = pd.DataFrame(master_results)

    # Save CSV
    csv_path = CREDE_OUT_DIR / "CreDE_Master_Summary.csv"
    df_summary.to_csv(csv_path, index=False)
    print(f"\nSaved CSV summary → {csv_path}")
    print(df_summary.to_string(index=False))

    # Credal coverage tables (overall + class-wise)
    credal_overall_df, credal_classwise_df = credal_frames(credal_records)
    credal_overall_df.to_csv(CREDE_OUT_DIR / 'CreDE_Credal_Coverage_Overall.csv', index=False)
    credal_classwise_df.to_csv(CREDE_OUT_DIR / 'CreDE_Credal_Coverage_ClassWise.csv', index=False)
    print('\nCreDE credal coverage (overall):')
    print(credal_overall_df.to_string(index=False))
    print('\nCreDE credal coverage (class-wise):')
    print(credal_classwise_df.to_string(index=False))

    # Pixel-level AU / EU / TU tables + threshold analysis (one sheet per model)
    print('Building uncertain-pixel tables...')
    pixel_uncertainty = {
        m: prepare_pixel_uncertainty(m, unc, y_img, CREDE_OUT_DIR)
        for m, unc in scene_uncertainty.items()
    }

    # Save Excel Report
    create_crede_excel_report(
        out_dir=CREDE_OUT_DIR,
        summary_df=df_summary,
        plot_paths=plot_entries,
        credal_overall_df=credal_overall_df,
        credal_classwise_df=credal_classwise_df,
        pixel_uncertainty=pixel_uncertainty,
    )
else:
    print("\n⚠️ No ensemble models found — generating empty summary report.")
    df_empty = pd.DataFrame(columns=["Model", "Mean_AU", "Mean_EU", "Mean_TU", "Credal_Coverage", "Credal_Avg_Set_Size"])
    df_empty.to_csv(CREDE_OUT_DIR / "CreDE_Master_Summary.csv", index=False)
    create_crede_excel_report(
        out_dir=CREDE_OUT_DIR,
        summary_df=df_empty,
        plot_paths=plot_entries,
        credal_overall_df=pd.DataFrame(columns=["Model", "N_Test_Pixels", "Overall_Empirical_Coverage", "Average_Set_Size", "Mean_Class_Coverage"]),
        credal_classwise_df=pd.DataFrame(columns=["Model"] + [f"Class_{k+1}_Coverage" for k in range(num_classes)]),
        pixel_uncertainty={},
    )



  Evaluating CreDE: AlexNet_CNN
  -> Loading & predicting: AlexNet_CNN_ens_1_final.keras
  -> Loading & predicting: AlexNet_CNN_ens_2_final.keras
  -> Loading & predicting: AlexNet_CNN_ens_3_final.keras
  -> Loading & predicting: AlexNet_CNN_ens_4_final.keras
  -> Loading & predicting: AlexNet_CNN_ens_5_final.keras
  -> Computing credal bounds...
  -> Credal coverage: 0.9993 | avg set size: 1.007
  -> Generating 3x4 spatial maps for AlexNet_CNN...
  -> Saved: /kaggle/working/Classification/ensemble/results/AlexNet_CNN_CreDE_spatial_maps.png

  Evaluating CreDE: GFNet
  -> Loading & predicting: GFNet_ens_1_final.keras
  -> Loading & predicting: GFNet_ens_2_final.keras
  -> Loading & predicting: GFNet_ens_3_final.keras
  -> Loading & predicting: GFNet_ens_4_final.keras
  -> Loading & predicting: GFNet_ens_5_final.keras
  -> Computing credal bounds...
  -> Credal coverage: 0.9993 | avg set size: 1.014
  -> Generating 3x4 spatial maps for GFNet...
  -> Saved: /kaggle/working/Classificatio